# Training on thin previews

At deployment some events have no competition preview, and we substitute our own cheap preview (`preview_agent`, "cheap" preset), which is much thinner. A model trained only on competition previews may misjudge those. Generating our own previews for past events would leak their results, so instead every competition preview is **condensed** into a thin one in the style of our real cheap preview (`suprise_dim.thin_preview`; Claude Haiku 5.5, low effort). All facts come from the competition's preview, written before the cutoff, so nothing leaks.

1. Inspect one condensed preview.
2. Condense all 1,365 preview events (train and test).
3. Run the same surprise extraction and v2 classification on the thin previews.
4. Build event features and compare thin with full previews.
5. Cross-validate three ridge models on train: **full-only** (today's model), **thin-only**, and **pooled** (both, with a preview-source flag and interactions). The key question: which predicts best from thin previews, without hurting full ones?
6. Score test once.

Cost of the full run: about 1,365 Haiku 5.5 calls (roughly $0.001 each), then about 1,365 extraction calls and about 260 classification calls (gpt-6-luna). Each step resumes after an interruption.

In [1]:
%load_ext autoreload
%autoreload 2

import re

import numpy as np
import pandas as pd

from ml import metrics, model, preprocess
from suprise_dim import data, dataset, extraction, features, grouping
from suprise_dim import thin_preview as tp

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

NAME = "thin_v1"                 # data/surprise/thin_previews_thin_v1.jsonl and thin_v1_extractions.jsonl
ASSIGN = f"v2_{NAME}"            # data/surprise/taxonomy/assignments_v2_thin_v1.jsonl
ES = model.ES

split = data.load_split()
events = data.load_eligible_events()
events_by_id = {e["event_id"]: e for e in events}
tax = grouping.load_taxonomy("v2")
labels = {e: "train" for e in split["train"]} | {e: "test" for e in split["test"]}
feat_full_all = pd.read_csv(features.features_path("v2"))
print(f"{len(events)} preview events: train {len(split['train'])}, test {len(split['test'])}")

1365 preview events: train 1076, test 289


## 1. One condensed preview

In [2]:
sample = events_by_id[split["train"][0]]
result = tp.thin_preview(sample)
full = data.items_by_id(sample)[data.PREVIEW_ID]["content"]
print(f"full: {len(full)} chars | thin: {len(result['preview'])} chars | tokens in/out: {result['input_tokens']}/{result['output_tokens']}")
print(result["preview"])

full: 9379 chars | thin: 4204 chars | tokens in/out: 7074/2839
# Twist Bioscience (TWST) Q3 FY2026 Earnings Preview

**Company:** Twist Bioscience (NASDAQ: TWST)
**Quarter:** Q3 FY2026 (quarter ended June 30, 2026; fiscal year ends Sept. 30)
**Earnings date:** Monday, Aug. 3, 2026, before market open; call at 8:00 a.m. ET. Source: Twist scheduling release, July 15, 2026.

**Data caveat:** Knowledge cutoff is July 31, 2026, 20:00 UTC. Figures below come from the May 4, 2026 Q2 results, the May 21 investor day, and third-party data dated July 16 to July 27, 2026. They may have changed since.

## 1. Consensus vs. Company Guidance

| Metric (Q3 FY26) | Company guidance | Consensus | Notes |
|---|---|---|---|
| Revenue | $114–115M (given May 4, 2026) | $114.55M (MarketBeat/Ticker Report, July 27, 2026) | Consensus sits near the guidance midpoint |
| EPS | Not guided | –$0.50 (MarketBeat/Ticker Report, July 27, 2026) | Unclear whether GAAP or adjusted |
| Gross margin | FY26 target >52% | No

## 2. Condense every preview

`N` counts events: start small, then `None` for all 1,365.

In [3]:
N = None  # None for every event
tp.run_thin_previews(events[:N] if N else events, NAME, workers=8)

Condensing 36 previews (1329 already done) with claude-haiku-5-5...
[1/36] ea_PHR_Q2_2027: 4923 chars
[2/36] ea_NTAP_Q1_2027: 4927 chars
[3/36] ea_MEI_Q1_2027: 4415 chars
[4/36] ea_LULU_Q2_2026: 5040 chars
[5/36] ea_MAMA_Q2_2027: 5405 chars
[6/36] ea_ZS_Q4_2026: 5165 chars
[7/36] ea_SNOW_Q2_2027: 6115 chars
[8/36] ea_OXM_Q2_2026: 5922 chars
[9/36] ea_AOUT_Q1_2027: 3817 chars
[10/36] ea_BBCP_Q3_2026: 4639 chars
[11/36] ea_GWRE_Q4_2026: 4886 chars
[12/36] ea_DOCU_Q2_2027: 4929 chars
[13/36] ea_CNM_Q2_2026: 4471 chars
[14/36] ea_EGAN_Q4_2026: 4728 chars
[15/36] ea_JMKE_Q2_2026: 5736 chars
[16/36] ea_PATH_Q2_2027: 4976 chars
[17/36] ea_IOT_Q2_2027: 6038 chars
[18/36] ea_CGNT_Q2_2027: 5586 chars
[19/36] ea_NNOX_Q2_2026: 4725 chars
[20/36] ea_DXLG_Q2_2026: 5650 chars
[21/36] ea_M_Q2_2026: 4337 chars
[22/36] ea_SAIL_Q2_2027: 5230 chars
[23/36] ea_DBI_Q2_2026: 4526 chars
[24/36] ea_LOVE_Q2_2027: 5752 chars
[25/36] ea_ODD_Q2_2026: 4877 chars
[26/36] ea_LSAK_Q4_2026: 4481 chars
[27/36] ea_FLWS_Q

{'requested': 1365, 'skipped': 1329, 'written': 36, 'failed': []}

In [4]:
# Does the thin version look like our real cheap preview?
thin = tp.load_thin_previews(NAME)
checks = {
    "chars": lambda t: len(t),
    "price stated": lambda t: bool(re.search(r"last price|closed at|market cap", t, re.I)),
    "implied move given": lambda t: bool(re.search(r"implied (earnings )?move[^\n]{0,80}\d+(\.\d+)?\s?%", t, re.I)),
    "consensus table": lambda t: "|" in t and "consensus" in t.lower(),
}
sets = {
    "competition": [data.items_by_id(events_by_id[e])[data.PREVIEW_ID]["content"] for e in thin],
    "thin (synthetic)": list(thin.values()),
    "real cheap (AEYE)": [tp.EXAMPLE_PATH.read_text()],
}
pd.DataFrame({name: {k: (np.median([f(t) for t in texts]) if k == "chars" else np.mean([f(t) for t in texts])) for k, f in checks.items()}
              for name, texts in sets.items()}).round(2)

,competition,thin (synthetic),real cheap (AEYE)
chars,10330.00,5113.00,5875.0
price stated,0.71,0.28,0.0
implied move given,0.49,0.27,0.0
consensus table,0.99,1.00,1.0


## 3. Extract and classify on the thin previews

The same extraction (preview prompt, gpt-6-luna medium) and v2 classification as for the full previews; only the preview text differs.

In [5]:
thin = tp.load_thin_previews(NAME)
thin_events = [tp.with_preview(events_by_id[e], text) for e, text in thin.items()]
extraction.run_extraction(thin_events, out_path=data.extractions_path(NAME))

records = [r for r in extraction.load_extractions(data.extractions_path(NAME)) if r["event_id"] in thin]
grouping.assign_items(grouping.surprise_items(records), tax, ASSIGN)

Extracting 1358 events (7 already done) with gpt-6-luna, preview prompt...
[1/1358] ea_TSN_Q3_2026: 7 surprises
[2/1358] ea_COHN_Q2_2026: 5 surprises
[3/1358] ea_PLOW_Q2_2026: 8 surprises
[4/1358] ea_MATX_Q2_2026: 7 surprises
[5/1358] ea_ADEA_Q2_2026: 7 surprises
[6/1358] ea_HESM_Q2_2026: 5 surprises
[7/1358] ea_AEIS_Q2_2026: 15 surprises
[8/1358] ea_TGTX_Q2_2026: 7 surprises
[9/1358] ea_AMRC_Q2_2026: 6 surprises
[10/1358] ea_PAY_Q2_2026: 4 surprises
[11/1358] ea_VRTX_Q2_2026: 7 surprises
[12/1358] ea_INSP_Q2_2026: 9 surprises
[13/1358] ea_PLTR_Q2_2026: 12 surprises
[14/1358] ea_CBT_Q3_2026: 8 surprises
[15/1358] ea_ALSN_Q2_2026: 12 surprises
[16/1358] ea_SNAP_Q2_2026: 10 surprises
[17/1358] ea_BWXT_Q2_2026: 8 surprises
[18/1358] ea_RVTY_Q2_2026: 9 surprises
[19/1358] ea_ARVN_Q2_2026: 7 surprises
[20/1358] ea_CIFR_Q2_2026: 8 surprises
[21/1358] ea_DD_Q2_2026: 7 surprises
[22/1358] ea_EVEX_Q2_2026: 10 surprises
[23/1358] ea_KMB_Q2_2026: 8 surprises
[24/1358] ea_FWRG_Q2_2026: 10 surprise

{'requested': 10270,
 'skipped': 0,
 'written': 10269,
 'missing': 1,
 'failed_batches': 0}

## 4. Features: thin vs full previews (same events)

In [6]:
assigned = set(grouping.load_assignments(ASSIGN)["item_id"])
records = [r for r in extraction.load_extractions(data.extractions_path(NAME)) if r["event_id"] in labels]
complete = {r["event_id"] for r in records if all(f"{r['event_id']}#{i}" in assigned for i in range(len(r["surprises"])))}

surprises_thin = dataset.build_extraction_frame(NAME, ASSIGN, {e: labels[e] for e in complete})
split_like = {"train": [e for e in split["train"] if e in complete], "test": [e for e in split["test"] if e in complete]}
feat_thin = features.build_event_features(surprises_thin, split_like, events_by_id, tax)
feat_full = feat_full_all[feat_full_all["event_id"].isin(complete)].sort_values("event_id").reset_index(drop=True)
feat_thin = feat_thin.sort_values("event_id").reset_index(drop=True)
assert (feat_full["event_id"] == feat_thin["event_id"]).all()
print(f"{len(complete)} events with a complete thin extraction: train {len(split_like['train'])}, test {len(split_like['test'])}")

pd.DataFrame({
    "full": feat_full[["n_items", "n_pos", "n_neg", "no_surprises"]].mean(),
    "thin": feat_thin[["n_items", "n_pos", "n_neg", "no_surprises"]].mean(),
}).round(2)

1364 events with a complete thin extraction: train 1075, test 289


,full,thin
n_items,7.72,7.52
n_pos,4.54,4.57
n_neg,2.18,1.95
no_surprises,0.01,0.01


In [7]:
# Single scores on train: how much does the thin preview lose?
tr = feat_full["split"] == "train"
y, es = feat_full.loc[tr, "target_pct"], feat_full.loc[tr, ES]
singles = {f"{s} {name}": model.rank_to_percentile(f.loc[tr, s]) for name, f in (("full", feat_full), ("thin", feat_thin)) for s in ("net_w", "net_w_first3")}
display(metrics.evaluate(y, singles, es).round(4))
print("rank correlation of net_w, full vs thin:", round(metrics.spearman(feat_full.loc[tr, "net_w"], feat_thin.loc[tr, "net_w"]), 3))

,n,r2_base,r2_full,delta_r2,spearman
net_w full,1027.0,0.073,0.2411,0.1681,0.4677
net_w_first3 full,1027.0,0.073,0.2359,0.1629,0.4658
net_w thin,1027.0,0.073,0.2025,0.1295,0.4197
net_w_first3 thin,1027.0,0.073,0.1934,0.1205,0.4119


rank correlation of net_w, full vs thin: 0.87


## 5. Three models, cross-validated on train

- **full-only**: today's approach, trained on full-preview features. Its "transfer" score is what it achieves when fed thin-preview features.
- **thin-only**: trained on thin-preview features.
- **pooled**: both versions of every train event, with `src_own` and interactions (`preprocess.add_source_features`) so it can weight thin previews differently. Both versions of an event always fall in the same fold.

All use the same grid and pooled out-of-fold ΔR² selection as `ridge_model.ipynb`.

In [8]:
train_full = preprocess.add_source_features(feat_full[tr].reset_index(drop=True), "competition")
train_thin = preprocess.add_source_features(feat_thin[tr].reset_index(drop=True), "own")
pooled = pd.concat([train_full, train_thin], ignore_index=True)

alphas = np.logspace(-1, 4, 11)
sets_plain = preprocess.feature_sets(feat_full.columns)
sets_pooled = {k: preprocess.with_source(v, pooled.columns) for k, v in sets_plain.items()}
grids = {
    "full-only": model.grid_search(train_full, sets_plain, alphas),
    "thin-only": model.grid_search(train_thin, sets_plain, alphas),
    "pooled": model.grid_search(pooled, sets_pooled, alphas),
}
best = {name: g.iloc[0] for name, g in grids.items()}
cols = ["feature_set", "alpha", "target_mode", "log_counts", "oof_delta_r2", "oof_spearman"]
pd.DataFrame({name: b[cols] for name, b in best.items()}).T

,feature_set,alpha,target_mode,log_counts,oof_delta_r2,oof_spearman
full-only,all,1000.0,raw,False,0.182548,0.490071
thin-only,all,1000.0,raw,False,0.146059,0.448362
pooled,all,3162.27766,raw,False,0.167067,0.472122


In [9]:
def oof(frame, name, sets, predict_on=None):
    b = best[name]
    out, _ = model.out_of_fold(frame, sets[b["feature_set"]], b["alpha"], target_mode=b["target_mode"],
                               log_counts=bool(b["log_counts"]), predict_on=predict_on)
    return out

pooled_oof = oof(pooled, "pooled", sets_pooled)
on_thin = pd.DataFrame({
    "full-only (transfer)": oof(train_full, "full-only", sets_plain, predict_on=train_thin).set_index("event_id")["oof"],
    "thin-only": oof(train_thin, "thin-only", sets_plain).set_index("event_id")["oof"],
    "pooled": pooled_oof[pooled_oof["preview_source"] == "own"].set_index("event_id")["oof"],
}).dropna()
on_full = pd.DataFrame({
    "full-only": oof(train_full, "full-only", sets_plain).set_index("event_id")["oof"],
    "pooled": pooled_oof[pooled_oof["preview_source"] == "competition"].set_index("event_id")["oof"],
}).dropna()

for title, table, ref in (("Predicting from THIN previews", on_thin, "pooled"), ("Predicting from FULL previews", on_full, "pooled")):
    rows = train_full.set_index("event_id").loc[table.index]
    preds = {c: table[c].to_numpy() for c in table}
    print(f"\n{title} ({len(table)} train events, out of fold)")
    display(metrics.evaluate(rows["target_pct"], preds, rows[ES]).round(4))
    display(metrics.compare_to(metrics.paired_bootstrap(rows["target_pct"], preds, rows[ES], n_boot=2000), ref).round(4))


Predicting from THIN previews (891 train events, out of fold)


,n,r2_base,r2_full,delta_r2,spearman
pooled,847.0,0.0767,0.2307,0.1540,0.4549
full-only (transfer),847.0,0.0767,0.2287,0.1520,0.4495
thin-only,847.0,0.0767,0.2228,0.1461,0.4484


,mean_diff,ci_low,ci_high,P(pooled better)
full-only (transfer),0.0020,-0.0042,0.0085,0.7255
thin-only,0.0079,0.0006,0.0154,0.9845



Predicting from FULL previews (891 train events, out of fold)


,n,r2_base,r2_full,delta_r2,spearman
full-only,847.0,0.0767,0.2593,0.1825,0.4901
pooled,847.0,0.0767,0.2582,0.1815,0.4900


,mean_diff,ci_low,ci_high,P(pooled better)
full-only,-0.0011,-0.007,0.0049,0.3555


## 6. Test, once

Each model is fitted on all train rows and scored on the test events, from thin previews and from full previews, on the same events, next to the competition baselines (which do not depend on our previews).

In [10]:
def fit_best(frame, name, sets):
    b = best[name]
    return model.fit(frame, sets[b["feature_set"]], b["alpha"], target_mode=b["target_mode"], log_counts=bool(b["log_counts"]))

te = feat_full["split"] == "test"
test_full = preprocess.add_source_features(feat_full[te].reset_index(drop=True), "competition")
test_thin = preprocess.add_source_features(feat_thin[te].reset_index(drop=True), "own")
fitted = {"full-only": fit_best(train_full, "full-only", sets_plain), "thin-only": fit_best(train_thin, "thin-only", sets_plain),
          "pooled": fit_best(pooled, "pooled", sets_pooled)}
BASELINES = sorted(c for c in test_full.columns if c.startswith("baseline__"))
common = test_full[[ES] + BASELINES].notna().all(axis=1).to_numpy()
y, es = test_full["target_pct"][common], test_full[ES][common]
base = {b.split("__")[1]: test_full[b].to_numpy()[common] for b in BASELINES}

for title, frame, names, ref in (("THIN previews", test_thin, ["full-only", "thin-only", "pooled"], "pooled"),
                                 ("FULL previews", test_full, ["full-only", "pooled"], "pooled")):
    preds = {n: fitted[n].predict(frame)[common] for n in names} | base
    print(f"\nTest, predicting from {title} ({common.sum()} events)")
    display(metrics.evaluate(y, preds, es).round(4))
    display(metrics.compare_to(metrics.paired_bootstrap(y, preds, es, n_boot=2000), ref).round(4))


Test, predicting from THIN previews (256 events)


,n,r2_base,r2_full,delta_r2,spearman
thin-only,256.0,0.1384,0.2583,0.1200,0.4284
pooled,256.0,0.1384,0.2548,0.1164,0.4166
full-only,256.0,0.1384,0.2518,0.1134,0.4049
z-ai,256.0,0.1384,0.2088,0.0704,0.3022
gemini,256.0,0.1384,0.2055,0.0672,0.3005
moonshotai,256.0,0.1384,0.1972,0.0589,0.2939
openai,256.0,0.1384,0.1898,0.0514,0.2741
deepseek,256.0,0.1384,0.1840,0.0457,0.2811


,mean_diff,ci_low,ci_high,P(pooled better)
full-only,0.0030,-0.0033,0.0097,0.8170
thin-only,-0.0035,-0.0111,0.0042,0.1725
deepseek,0.0697,0.0201,0.1260,0.9985
gemini,0.0481,-0.0000,0.1038,0.9745
moonshotai,0.0567,0.0101,0.1101,0.9875
openai,0.0634,0.0146,0.1200,0.9955
z-ai,0.0450,-0.0137,0.1042,0.9420



Test, predicting from FULL previews (256 events)


,n,r2_base,r2_full,delta_r2,spearman
pooled,256.0,0.1384,0.2726,0.1343,0.4618
full-only,256.0,0.1384,0.2674,0.1290,0.4542
z-ai,256.0,0.1384,0.2088,0.0704,0.3022
gemini,256.0,0.1384,0.2055,0.0672,0.3005
moonshotai,256.0,0.1384,0.1972,0.0589,0.2939
openai,256.0,0.1384,0.1898,0.0514,0.2741
deepseek,256.0,0.1384,0.1840,0.0457,0.2811


,mean_diff,ci_low,ci_high,P(pooled better)
full-only,0.0050,-0.0016,0.0123,0.9290
deepseek,0.0875,0.0353,0.1483,1.0000
gemini,0.0658,0.0131,0.1248,0.9925
moonshotai,0.0745,0.0225,0.1355,0.9980
openai,0.0812,0.0238,0.1463,0.9990
z-ai,0.0627,0.0028,0.1287,0.9785


### Reading the result

- **Pooled beats full-only (transfer) on thin previews and roughly ties it on full previews:** use the pooled model with a preview-source flag at deployment.
- **Thin-only is close to pooled on thin previews:** a separate model per source is equally good and simpler.
- **No model gets much from thin previews:** the extra Agent SDK run may not be worth it over the no-preview branch; compare with the no-preview results in `no_preview.ipynb`.

Synthetic thin previews are a stand-in. Real cheap previews collected before the cutoff (shadow mode) are the final check.